# This notebook demonstrates a series of experiments conducted with ADAg on the AI4I-2021 dataset

## Step 0. Setup the experiment

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
import os
import pandas as pd
import numpy as np
from adag.models import AnomalyDetectorPro
from support.evaluate import evaluate
from spag.models import DeepForecaster
from support.preproc import load_reg_windows

## Step 1. Load data 

### Train and Test

In [23]:
train = {}
test = {}
data_folder = os.path.join('data', 'spag', 'input')
for state in os.listdir(data_folder):
    state_path = os.path.join(data_folder, state)
    train[state] = load_reg_windows(os.path.join(state_path, 'train'))
    print('train', state,  'data shape:', train[state].data.shape, 'targets shape:', train[state].targets.shape)
    test[state] = load_reg_windows(os.path.join(state_path, 'test'))
    print('test', state,  'data shape:', test[state].data.shape, 'targets shape:', test[state].targets.shape)

train c25-75 data shape: (2800, 60, 3) targets shape: (2800, 15, 3)
test c25-75 data shape: (700, 60, 3) targets shape: (700, 15, 3)
train cap data shape: (2800, 60, 3) targets shape: (2800, 15, 3)
test cap data shape: (700, 60, 3) targets shape: (700, 15, 3)
train off data shape: (2800, 60, 3) targets shape: (2800, 15, 3)
test off data shape: (700, 60, 3) targets shape: (700, 15, 3)
train on data shape: (2800, 60, 3) targets shape: (2800, 15, 3)
test on data shape: (700, 60, 3) targets shape: (700, 15, 3)
train out data shape: (2800, 60, 3) targets shape: (2800, 15, 3)
test out data shape: (700, 60, 3) targets shape: (700, 15, 3)
train unb data shape: (2800, 60, 3) targets shape: (2800, 15, 3)
test unb data shape: (700, 60, 3) targets shape: (700, 15, 3)
train vnt data shape: (2800, 60, 3) targets shape: (2800, 15, 3)
test vnt data shape: (700, 60, 3) targets shape: (700, 15, 3)


In [24]:
states = ['on', 'cap', 'out', 'unb', 'c25-75', 'vnt']

In [25]:
test_trues = np.concatenate([test[state].targets for state in states])
test_true_labels = []
for state in states:
    y = 0 if state == 'on' else 1
    test_true_labels.extend([y for _ in range(len(test[state].targets))])

## Step 2. Load models

### Normal models

In [26]:
model_directory = os.path.join('models', 'spag')
models = {}

In [27]:
normal_state = '02_on'
model_names = os.listdir(os.path.join(model_directory, normal_state))

for model_name in model_names:
    model_path = os.path.join(model_directory, normal_state, model_name)
    model = DeepForecaster(from_file=model_path)
    model.default_name = f"{normal_state}_{model_name.replace('.keras', '')}"
    model.model._name = model.default_name
    print(model.summary())
    models[model_name.replace('.keras', '')] = model

Model: "02_on_GRU-1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru_6 (GRU)                 (None, 32)                3552      
                                                                 
 dropout_13 (Dropout)        (None, 32)                0         
                                                                 
 dense_10 (Dense)            (None, 45)                1485      
                                                                 
 reshape_10 (Reshape)        (None, 15, 3)             0         
                                                                 
Total params: 5037 (19.68 KB)
Trainable params: 5037 (19.68 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None
Model: "02_on_GRU-3"
_________________________________________________________________
 Layer (type)                Output Shape              Param 

## Step 3. Forecasting

In [28]:
test_preds = {model_name: [] for model_name in model_names}
for model_name, model in models.items():
    print(model_name)
    test_preds[model_name] = np.concatenate([model.forecasting(data=test[state].data) for state in states])

GRU-1
GRU-3
LSTM-1


### Create detectors

In [14]:
detectors = {}
percentiles = {'GRU-1': 0.95, 'LSTM-1': 0.99, 'GRU-3': 0.95}
for model_name, model in models.items():
    print(model_name)
    detector = AnomalyDetectorPro(classes={0: [0, 1], 1: [2, 3, 4, 5, 6]},
                                  percentile=percentiles[model_name])
    detector.train(true=train['on'].targets,
                   pred=model.forecasting(data=train['on'].data))
    detectors[model_name] = detector
    print('Detector parameters:', detector.__dict__)

GRU-1
Detector parameters: {'percentile': 0.95, 'classes': {0: [0, 1], 1: [2, 3, 4, 5, 6]}, 'alpha': 0.4, 'beta': 0.3, 'gamma': 0.2, 'shape': 11.38335345096997, 'loc': 0, 'scale': 0.0017233167373576922}
GRU-3
Detector parameters: {'percentile': 0.95, 'classes': {0: [0, 1], 1: [2, 3, 4, 5, 6]}, 'alpha': 0.4, 'beta': 0.3, 'gamma': 0.2, 'shape': 13.136853196946907, 'loc': 0, 'scale': 0.0016640643986628716}
LSTM-1
Detector parameters: {'percentile': 0.99, 'classes': {0: [0, 1], 1: [2, 3, 4, 5, 6]}, 'alpha': 0.4, 'beta': 0.3, 'gamma': 0.2, 'shape': 11.862660715040196, 'loc': 0, 'scale': 0.0015849382606031275}


In [29]:
path_to_save = os.path.join('models', 'adag')
if not os.path.exists(path_to_save):
    os.makedirs(path_to_save)

for model_name, detector in detectors.items():
    detector.save(os.path.join(path_to_save, model_name))

### Get data from SCAg agenets (C-SCAg and P-SCAg)

In [30]:
probs_path = os.path.join('data', 'spag', 'results', 'probs.csv')
scag_results = pd.read_csv(probs_path)
scag_results.head()

,pred_model,clf_model,real_class,test_class,pred_class,t_class_0_prob,p_class_0_prob,t_class_1_prob,p_class_1_prob,t_class_2_prob,p_class_2_prob,t_class_3_prob,p_class_3_prob,t_class_4_prob,p_class_4_prob,t_class_5_prob,p_class_5_prob,t_class_6_prob,p_class_6_prob
0,gru_32_16_d01,ParametrizedAE,0,0,0,0.999855,0.999854,0.000124,0.000124,1.068269e-09,1.077933e-09,4.143043e-07,4.225812e-07,3.263864e-09,3.318811e-09,9.637343e-08,9.815711e-08,0.000021,0.000021
1,gru_32_16_d01,ParametrizedAE,0,0,0,0.999853,0.999855,0.000125,0.000124,1.084746e-09,1.077985e-09,4.207549e-07,4.233912e-07,3.280854e-09,3.324317e-09,9.708506e-08,9.832614e-08,0.000021,0.000021
2,gru_32_16_d01,ParametrizedAE,0,0,0,0.999852,0.999855,0.000126,0.000123,1.099736e-09,1.076249e-09,4.255790e-07,4.239311e-07,3.305984e-09,3.330023e-09,9.772685e-08,9.850130e-08,0.000021,0.000021
3,gru_32_16_d01,ParametrizedAE,0,0,0,0.999855,0.999855,0.000123,0.000123,1.066038e-09,1.077029e-09,4.170708e-07,4.245194e-07,3.304790e-09,3.332863e-09,9.746802e-08,9.858889e-08,0.000021,0.000021
4,gru_32_16_d01,ParametrizedAE,0,0,0,0.999852,0.999855,0.000126,0.000124,1.098352e-09,1.078468e-09,4.262143e-07,4.236768e-07,3.315374e-09,3.326143e-09,9.797096e-08,9.839181e-08,0.000021,0.000021


## Step 4. Anomaly detection

In [31]:
anomaly_detection_results = {}
for model_name, detector in detectors.items():
    anomaly_detection_results[model_name] = {}
    for clf_model in ['ParametrizedMLP', 'ParametrizedCNN', 'ParametrizedRNN']:
        print(model_name, clf_model)
        anomaly_detection_results[model_name][clf_model] = {}
        df = scag_results[(scag_results['pred_model'] == model_name) & (scag_results['real_class'] > 0) & (
                scag_results['clf_model'] == clf_model)]
        anomaly_detection_results[model_name][clf_model]['simple'] = (
            detector.simply_run(true_data=test_trues, pred_data=test_preds[model_name]))
        anomaly_detection_results[model_name][clf_model]['corrected'] = (
            detector.run(true_data=test_trues, pred_data=test_preds[model_name],
                         current_labels=df['test_class'].values, pred_labels=df['pred_class'].values,
                         current_proba=df[[c for c in df.columns if ('t_class' in c) and ('prob' in c)]].values,
                         pred_proba=df[[c for c in df.columns if ('p_class' in c) and ('prob' in c)]].values))

GRU-1 ParametrizedMLP
GRU-1 ParametrizedCNN
GRU-1 ParametrizedRNN
GRU-3 ParametrizedMLP
GRU-3 ParametrizedCNN
GRU-3 ParametrizedRNN
LSTM-1 ParametrizedMLP
LSTM-1 ParametrizedCNN
LSTM-1 ParametrizedRNN


### Connected with Table 9

In [32]:
results = pd.DataFrame(columns=['pred_model', 'clf_model', 'detection_mode', 'ACC', 'P', 'R', 'F1', 'FNR', 'FPR'])
i = 0
for reg_model, reg_val in anomaly_detection_results.items():
    for clf_model, val in reg_val.items():
        for mode in ['simple', 'corrected']:
            results.loc[i, 'pred_model'] = reg_model
            results.loc[i, 'clf_model'] = clf_model
            results.loc[i, 'detection_mode'] = mode
            evals = evaluate(test_true_labels, val[mode]['labels'], balanced=True)
            results.loc[i, ['ACC', 'P', 'R', 'F1', 'FNR', 'FPR']] = list(evals.values())
            i = i + 1
results

,pred_model,clf_model,detection_mode,ACC,P,R,F1,FNR,FPR
0,GRU-1,ParametrizedMLP,simple,0.912,0.857,0.99,0.918,0.166,0.01
1,GRU-1,ParametrizedMLP,corrected,0.5,1.0,0.0,0.0,0.0,1.0
2,GRU-1,ParametrizedCNN,simple,0.911,0.856,0.987,0.917,0.166,0.013
3,GRU-1,ParametrizedCNN,corrected,0.5,1.0,0.0,0.0,0.0,1.0
4,GRU-1,ParametrizedRNN,simple,0.911,0.856,0.989,0.918,0.166,0.011
5,GRU-1,ParametrizedRNN,corrected,0.5,1.0,0.0,0.0,0.0,1.0
6,GRU-3,ParametrizedMLP,simple,0.923,0.882,0.976,0.927,0.13,0.024
7,GRU-3,ParametrizedMLP,corrected,0.5,1.0,0.0,0.0,0.0,1.0
8,GRU-3,ParametrizedCNN,simple,0.919,0.882,0.967,0.922,0.13,0.033
9,GRU-3,ParametrizedCNN,corrected,0.5,1.0,0.0,0.0,0.0,1.0


## This concludes the demonstration for the AI4I-2021 dataset.